
# 🧪 Lab 2 — **LLM Fine-tuning** (DistilGPT2) + **LoRA** + **Prompting**

**Course:** Generative AI (Day 2)  
**Lab Length:** ~3 hours

**Goal:** Fine-tune a small causal LM on a tiny domain corpus (Tiny Shakespeare), compare **full fine-tuning** vs **LoRA**, and experiment with **prompt engineering**.

> ✅ **Deliverables (submit this notebook):**
> - Baseline vs fine-tuned **perplexity** table
> - 3–5 **generations** for the same prompts across models (baseline / full FT / LoRA)
> - Short **prompting analysis** (zero-shot, few-shot, instruction template)
> - 3–5 bullet **takeaways**



### 🚦 Rules & Guidance
- If `bitsandbytes` isn't available, LoRA still works without 4-bit (skip QLoRA).
- Cells marked **(Provided)** can be run as-is; **(TODO)** require your edits.



## 🎯 Learning Objectives
- Prepare data, tokenize, and **pack** sequences for causal LM training.
- Compute baseline **validation perplexity** (no fine-tuning).
- Run a short **full fine-tune** (a few epochs) and evaluate.
- Apply **LoRA (PEFT)** (optionally QLoRA) and evaluate.
- Compare **generations** and analyze **prompting** strategies.


In [10]:
# ===== (Provided) Setup & Installs =====
# Install current, mutually-compatible versions. (The old pinned versions,
# especially bitsandbytes==0.43.1, were broken in current Colab.)
!pip -q install -U "transformers==4.46.3" "peft==0.13.2" "accelerate==1.1.1" "datasets==3.1.0" "bitsandbytes==0.44.1"
!pip -q uninstall -y torchao

import os, math, random, numpy as np, torch
from datasets import load_dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling,
                          Trainer, TrainingArguments, set_seed)
from transformers import pipeline
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import evaluate

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda



---

## Part 0 — Data (Tiny Shakespeare) *(~10 min)*

We'll use the **Tiny Shakespeare** dataset (~1MB).  
Feel free to swap with a small domain corpus if desired.


In [2]:

# ===== (Provided) Load dataset =====
raw = load_dataset("Trelis/tiny-shakespeare")
# Split into train/valid/test quickly
split = raw["train"].train_test_split(test_size=0.02, seed=42)
test_valid = split["test"].train_test_split(test_size=0.5, seed=42)
dataset = DatasetDict({
    "train": split["train"],
    "validation": test_valid["train"],
    "test": test_valid["test"]
})
dataset


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/497 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/119k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/472 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/49 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Text'],
        num_rows: 462
    })
    validation: Dataset({
        features: ['Text'],
        num_rows: 5
    })
    test: Dataset({
        features: ['Text'],
        num_rows: 5
    })
})


---

## Part 1 — Tokenization & Packing *(~25–30 min)*

**Tasks:**
1) Load tokenizer for `distilgpt2` and set `pad_token = eos_token`.  
2) **Tokenize** the text.  
3) **Group/pack** into fixed-length blocks (e.g., `block_size = 256`) with `labels = input_ids`.

> 💡 **Hints**
> - Use `remove_columns=["text"]` in `map` after tokenization.
> - For grouping, concatenate token lists and then slice into blocks.


In [3]:

# ===== (TODO) Tokenizer & packing =====
model_id = "distilgpt2"
tok = AutoTokenizer.from_pretrained(model_id)
tok.pad_token = tok.eos_token  # important for collation
block_size = 256

def tokenize(ex):
    out = tok(ex["Text"])
    return out

tokenized = dataset.map(tokenize, batched=True, remove_columns=["Text"])
tokenized

def group_texts(examples):
    # - Concatenate lists per key
    # - Truncate to a multiple of block_size
    # - Split into fixed blocks
    concat = {k: sum(examples[k], []) for k in examples.keys()}
    total_len = (len(concat["input_ids"]) // block_size) * block_size
    result = {
        k: [t[i:i+block_size] for i in range(0, total_len, block_size)]
        for k, t in concat.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_datasets = tokenized.map(group_texts, batched=True)
lm_datasets


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Map:   0%|          | 0/462 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1406
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 15
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 17
    })
})


---

## Part 2 — Baseline Perplexity *(~20 min)*

Compute validation **perplexity** before any fine-tuning.

> 💡 **Hints**
> - Use `AutoModelForCausalLM` with `distilgpt2`.
> - Use `DataCollatorForLanguageModeling(mlm=False)` for causal LM.
> - Evaluate a limited number of batches (e.g., 50) for speed.


In [4]:

# ===== Baseline perplexity =====
baseline_model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
collator = DataCollatorForLanguageModeling(tok, mlm=False)

def compute_ppl(model, ds, max_batches=50):
    model.eval()
    losses = []
    loader = torch.utils.data.DataLoader(ds, batch_size=8, shuffle=False, collate_fn=collator)
    for i, batch in enumerate(loader):
        if i >= max_batches: break
        for k in batch:
            batch[k] = batch[k].to(device)
        with torch.no_grad():
            out = model(**batch)
            loss = out.loss.detach().float()
        losses.append(loss.item())
    mean_loss = float(np.mean(losses)) if losses else float("inf")
    return math.exp(mean_loss) if mean_loss < 20 else float("inf")

ppl_baseline = compute_ppl(baseline_model, lm_datasets["validation"])
print(f"Baseline validation PPL: {ppl_baseline:.2f}")


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Baseline validation PPL: 68.46



---

## Part 3 — Full Fine-tune *(~45–60 min)*

Run a short fine-tune (2–3 epochs).

> 💡 **Hints**
> - Start with `per_device_train_batch_size=8`, `grad_accum_steps=2` (adjust to VRAM).
> - `fp16=True` if on GPU.
> - Log with `logging_steps=50` and evaluate per epoch.


In [5]:
# ===== Full fine-tune =====
ft_model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

args = TrainingArguments(
    output_dir="./ft-distilgpt2",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",
    logging_steps=50,
    learning_rate=2e-4,
    num_train_epochs=5,
    warmup_ratio=0.03,
    weight_decay=0.1,
    save_strategy="epoch",
    fp16=False,        # fp32 training; avoids "unscale FP16 gradients" error
    report_to="none"
)

trainer = Trainer(
    model=ft_model,
    args=args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
    data_collator=collator,
)

trainer.train()

ppl_ft = compute_ppl(ft_model, lm_datasets["validation"])
print(f"Full FT validation PPL: {ppl_ft:.2f}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,3.969660,3.360661
2,3.440764,3.248597
3,3.252886,3.215062
4,3.119228,3.197006
5,3.046227,3.202818


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Full FT validation PPL: 24.93



---

## Part 4 — Quick Generations (Full FT) *(~10 min)*

Generate a short sample with your **fine-tuned** model for qualitative inspection.


In [6]:

# ===== (Provided) Generation helper for FT model =====
gen_ft = pipeline("text-generation", model=ft_model, tokenizer=tok, device=0 if device=="cuda" else -1)

prompt = "ROMEO: I dreamt tonight that"
out = gen_ft(prompt, max_new_tokens=80, do_sample=True, temperature=0.9, top_p=0.95)[0]["generated_text"]
print(out)


[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'top_p', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


ROMEO: I dreamt tonight that thou
should be the cause of her death, I cannot
think of what is meant to be it.

HERMIONE:
Come to me, cousin; come, cousin:
I hope I shall see you when I meet you at the Corioli:
Tell me what, shall I hear you that are true?

LADY ANNE:
What



---

## Part 5 — LoRA (PEFT) *(~45 min)*

Train with **LoRA** (and optionally QLoRA if 4-bit quantization is available).

> 💡 **Hints**
> - If `bitsandbytes` is available, use `BitsAndBytesConfig(load_in_4bit=True)` + `prepare_model_for_kbit_training`.
> - Set `r=8..16`, `lora_alpha=16..32`, `lora_dropout≈0.05`.
> - Compare PPL vs full FT.


In [11]:
# ===== LoRA fine-tune =====
# distilgpt2 is small -> no quantization/bitsandbytes needed.
base_for_lora = AutoModelForCausalLM.from_pretrained(model_id).to(device)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["c_attn"],   # correct attention module name for GPT-2/distilgpt2
    bias="none",
    task_type="CAUSAL_LM"
)
lora_model = get_peft_model(base_for_lora, lora_cfg)
lora_model.print_trainable_parameters()

args_lora = TrainingArguments(
    output_dir="./lora-distilgpt2",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",
    logging_steps=50,
    learning_rate=1.5e-4,
    num_train_epochs=2,
    warmup_ratio=0.03,
    save_strategy="epoch",
    fp16=False,
    report_to="none"
)

trainer_lora = Trainer(
    model=lora_model,
    args=args_lora,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
    data_collator=collator,
)
trainer_lora.train()

ppl_lora = compute_ppl(lora_model, lm_datasets["validation"])
print(f"LoRA validation PPL: {ppl_lora:.2f}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 294,912 || all params: 82,207,488 || trainable%: 0.3587


Epoch,Training Loss,Validation Loss
1,4.360562,3.917293
2,4.213452,3.873175


LoRA validation PPL: 48.70



---

## Part 6 — Prompting Experiments *(~30 min)*

Run **the same prompts** through:
- Baseline (no FT)
- Full FT
- LoRA

Explore:
- **Zero-shot** continuation
- **Instruction-style** template (even if GPT-2 family isn't instruction-tuned, observe behavior)
- **Few-shot** pattern completion


In [12]:
# ===== Compare generations across models =====
gen_base = pipeline("text-generation", model=baseline_model, tokenizer=tok, device=0 if device=="cuda" else -1)
gen_lora = pipeline("text-generation", model=lora_model,     tokenizer=tok, device=0 if device=="cuda" else -1)

prompts = [
    "ROMEO: I dreamt tonight that",
    "### Instruction:\nRewrite the line in modern English.\n### Input:\n'Thou art more lovely and more temperate.'\n### Response:\n",
    "SCENE PROMPTS:\nExample: 'A dark forest at midnight.' -> 'The moon hides as branches claw at the sky.'\n"
    "Example: 'A crowded marketplace.' -> 'Vendors roar; coins clatter like rain.'\n"
    "Task: 'A stormy seashore.' -> ",
]

def run_batch(gen_pipe, name):
    print(f"\n=== {name} ===")
    for p in prompts:
        out = gen_pipe(p, max_new_tokens=60, do_sample=True, temperature=0.9, top_p=0.92)[0]["generated_text"]
        print(f"\n[Prompt]\n{p}\n[Output]\n{out}\n{'-'*60}")

run_batch(gen_base, "Baseline")
run_batch(gen_ft,   "Full FT")
run_batch(gen_lora, "LoRA")

[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== Baseline ===


[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Prompt]
ROMEO: I dreamt tonight that
[Output]
ROMEO: I dreamt tonight that my dream to become a reality has been fulfilled.


I believe that God has created this dream of my life, as an opportunity to begin with. The world has a vision of that future, but it is beyond my power to choose. We are faced with the fate of our planet.
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


[Prompt]
SCENE PROMPTS:
Example: 'A dark forest at midnight.' -> 'The moon hides as branches claw at the sky.'
Example: 'A crowded marketplace.' -> 'Vendors roar; coins clatter like rain.'
Task: 'A stormy seashore.' -> 
[Output]
SCENE PROMPTS:
Example: 'A dark forest at midnight.' -> 'The moon hides as branches claw at the sky.'
Example: 'A crowded marketplace.' -> 'Vendors roar; coins clatter like rain.'
Task: 'A stormy seashore.' -> ipsingers-eye
Sicinies:
Virtue-tied clouds of fire; a deadly tempest-tide
With sparks of fire: a mortal eye
Shall be a scene of heaven, a scene of hell,
That God's sovereign will do unto
------------------------------------------------------------

=== LoRA ===


[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Prompt]
ROMEO: I dreamt tonight that
[Output]
ROMEO: I dreamt tonight that he had come and I will not sleep with you;

I will not sleep with you;
And not sleep with you.
BARTARD: If you had him in my room, and I must be with you.
BARTARD: I don't think you'd do much
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Prompt]
### Instruction:
Rewrite the line in modern English.
### Input:
'Thou art more lovely and more temperate.'
### Response:

[Output]
### Instruction:
Rewrite the line in modern English.
### Input:
'Thou art more lovely and more temperate.'
### Response:
'I like you too. But please,
'Thou art more charming and more temperate.'
###
------------------------------------------------------------

[Prompt]
SCENE PROMPTS:
Example: 'A dark forest at midnight.' -> 'The moon hides as branches claw at the sky.'
Example: 'A crowded marketplace.' -> 'Vendors roar; coins clatter like rain.'
Task: 'A stormy seashore.' -> 
[Output]
SCENE PROMPTS:
Example: 'A dark forest at midnight.' -> 'The moon hides as branches claw at the sky.'
Example: 'A crowded marketplace.' -> 'Vendors roar; coins clatter like rain.'
Task: 'A stormy seashore.' -> ------------
Example: 'A man's heart is ripped out of my heart' -> 'A man's heart is torn out of my heart.' -> 'A man's heart is ripped out of my heart.'
Task:


---

# Part 7 — Challenge: The "Catastrophic Forgetting" Test

**Context:** You have fine-tuned the model to speak like Shakespeare. But what happened to the general knowledge it had before?

When we aggressively fine-tune a small model on a narrow dataset, we often trigger **Catastrophic Forgetting**—the model "forgets" how to speak modern English or answer factual questions because its weights have been overwritten to minimize loss on Shakespeare plays.

**Task:**
1. Run the code below using a **modern** prompt (something that definitely didn't exist in Shakespeare's time).
2. Observe how the **Full Fine-Tune** model struggles compared to the **Baseline**.
3. **Crucial:** Analyze if **LoRA** preserves more general knowledge than Full FT.

In [13]:
# ===== The Forgetting Test =====
# Probe whether fine-tuning erased general (non-Shakespeare) knowledge.
# Build fresh pipelines from the trained models so this cell is self-contained.
_dev = 0 if device == "cuda" else -1
_gb = pipeline("text-generation", model=baseline_model, tokenizer=tok, device=_dev)
_gf = pipeline("text-generation", model=ft_model,       tokenizer=tok, device=_dev)
_gl = pipeline("text-generation", model=lora_model,     tokenizer=tok, device=_dev)

forget_prompt = "Explain how WiFi works:"

for gen, label in [(_gb, "BASELINE"), (_gf, "FULL FINE-TUNE"), (_gl, "LoRA")]:
    out = gen(forget_prompt, max_new_tokens=60, do_sample=True,
              temperature=0.9, top_p=0.95)[0]["generated_text"]
    print(f"=== {label} ===\n{out}\n")

[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== BASELINE ===
Explain how WiFi works:































































[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== FULL FINE-TUNE ===
Explain how WiFi works: there's no set time limit
For our speed; we'll be slow, when we'll find
A thousand words a day!

POMPEY:
Not only is he a dangerous man,
But he is not himself: it is so, he shall not be
An

=== LoRA ===
Explain how WiFi works:

Now on to the next time:
First thing I want to say for you to follow me:
I've always believed that when I am presented with an opportunity to enter a private sector, I am in a situation where I know that the value of the opportunity can be determined by a



---
# Part 8 — The "Ablation Study"

**Warning: This section requires significant compute time.**

**Context:**
In Part 5, you used a LoRA Rank (`r`) of **16**. But why 16? Why not 1? Why not 100?
In Deep Learning, hyperparameters are rarely guessed correctly on the first try. We must perform an **Ablation Study**—a systematic experiment where we change one variable to see its impact.

**The Theory:**
* **Rank (`r`)** determines the size of the low-rank matrices.
* **High Rank (e.g., 64):** More trainable parameters. Theoretically "smarter," but slower to train and higher risk of overfitting.
* **Low Rank (e.g., 1 or 8):** Fewer parameters. Faster, but might not have the "capacity" to learn the style.

**Your Task:**
You will run the LoRA training loop **3 distinct times** to find the "Sweet Spot."

1. **Run A:** `r = 1` (Extreme compression)
2. **Run B:** `r = 8` (Low capacity)
3. **Run C:** `r = 64` (High capacity)

*Note: To save time, you can reduce `num_train_epochs` to **1** for these experiments, as we are looking for relative differences, not absolute convergence.*

### 📝 Deliverable Table (Fill this out)

| Experiment | Rank (`r`) | Trainable Parameters (%) | Validation PPL | Did it learn the style? (Subjective) |
| :--- | :--- | :--- | :--- | :--- |
| **Run A** | 1 | ... % | ... | ... |
| **Run B** | 8 | ... % | ... | ... |
| **Run C** | 64 | ... % | ... | ... |



---
*(Run the code cell below to execute the loop. Go grab a coffee, this will take a while!)*

In [14]:
# ===== Ablation Study Loop =====
# This cell runs the training 3 times. (WARNING: takes a while.)

ranks_to_test = [1, 8, 64]
results = {}

for r_val in ranks_to_test:
    print(f"\n\n{'='*20} STARTING TRAINING WITH RANK: {r_val} {'='*20}")

    loop_config = LoraConfig(
        r=r_val,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["c_attn"],   # correct module name for distilgpt2
        bias="none",
        task_type="CAUSAL_LM"
    )

    model_to_train = AutoModelForCausalLM.from_pretrained(model_id).to(device)
    loop_model = get_peft_model(model_to_train, loop_config)

    trainable, total = loop_model.get_nb_trainable_parameters()
    print(f"Trainable params: {trainable} || {100 * trainable / total:.4f}%")

    loop_args = TrainingArguments(
        output_dir=f"./lora-rank-{r_val}",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=1.5e-4,
        num_train_epochs=1,
        logging_steps=50,
        fp16=False,
        report_to="none"
    )

    loop_trainer = Trainer(
        model=loop_model,
        args=loop_args,
        train_dataset=lm_datasets["train"],
        eval_dataset=lm_datasets["validation"],
        data_collator=collator
    )
    loop_trainer.train()

    ppl = compute_ppl(loop_model, lm_datasets["validation"])
    results[r_val] = ppl
    print(f"Rank {r_val} Finished. PPL: {ppl:.2f}")

    del loop_model, loop_trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n\n=== FINAL RESULTS ===")
print(results)



==================== STARTING TRAINING WITH RANK: 1 ====================


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.


Trainable params: 18432 || 0.0225%


Step,Training Loss
50,4.363751


...

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.


Trainable params: 1179648 || 1.4197%


Step,Training Loss
50,4.356769


Rank 64 Finished. PPL: 53.50


=== FINAL RESULTS ===
{1: 54.110715940656824, 8: 53.549179586612176, 64: 53.49783571251199}



---

## 📝 Final Report (fill before submission)

Complete the table and add brief commentary.

### Perplexity
| Model | Val PPL |
|------|---------|
| Baseline (distilgpt2) | … |
| Full Fine-tune | … |
| LoRA | … |

### Generations (same prompts)
- Prompt A:  
  - Baseline → …  
  - Full FT → …  
  - LoRA → …  

- Prompt B: …

### Prompt Engineering Notes
- Zero-shot vs few-shot: …  
- Instruction template effects: …  
- Failure cases / artifacts: …  

### Analysis: Catastrophic Forgetting
* **Observation:** How did the Full Fine-Tuned model respond to the modern prompt compared to the Baseline? Did it hallucinate Shakespearean words?
* **LoRA vs Full FT:** Did the LoRA model retain more "modern English" capability than the Full FT model?
* **Theory:** Why does LoRA (training <1% of parameters) theoretically help prevent forgetting compared to updating 100% of parameters?

### Analysis Questions: The "Ablation Study"

1. **The "Diminishing Returns" Trap:** Did increasing the Rank from 8 to 64 result in a massive drop in Perplexity, or was the improvement marginal?
2. **Efficiency:** Look at the file size or parameter count of `r=1` vs `r=64`. Given the performance difference you observed, which Rank would you choose for a production mobile app?
3. **Overfitting:** Did the high-rank model (`r=64`) start to memorize the training data (loss goes down) but fail to generalize (validation PPL stays high)? Explain based on your logs.

### Takeaways (3–5 bullets)
- …
